#  Шаг 1: Сформировать сопоставимую выборку и `slice.jsonl`

Из двух файлов:
- `alpaca_eval_annotations_alpaca_eval_gpt4.json`
- `alpaca_eval_annotations_claude.json`

отбираем только объекты, где совпадают инструкция и оба варианта ответа.

Замечание по обозначениям `preference`:
- `preference = 1.0` - предпочтительнее `output_1`
- `preference = 2.0` - предпочтительнее `output_2`
- `preference = 0.0` - оба ответа одинаковые (tie, ничья)

In [1]:
import pandas as pd
from datasets import load_dataset

BASE_URL = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/"
claude = load_dataset("json", data_files=BASE_URL+"alpaca_eval_annotations_claude.json", split="train").to_pandas()
gpt4   = load_dataset("json", data_files=BASE_URL+"alpaca_eval_annotations_alpaca_eval_gpt4.json", split="train").to_pandas()

KEY = ["instruction", "output_1", "output_2", "generator_1", "generator_2"]

# Проверяем попарное выравнивание строк
aligned = claude[KEY].reset_index(drop=True).equals(
    gpt4[KEY].reset_index(drop=True)
)
print("Строки выровнены по содержимому пары:", aligned)

if aligned:
    merged = claude.copy().reset_index(drop=True)
    merged["pref_claude"] = claude["preference"].reset_index(drop=True)
    merged["pref_gpt4"]   = gpt4["preference"].reset_index(drop=True)
else:
    # запасной вариант: соединение по ключу
    merged = claude.merge(
        gpt4[KEY + ["preference"]].rename(columns={"preference": "pref_gpt4"}),
        on=KEY, how="inner")
    merged = merged.rename(columns={"preference": "pref_claude"})

comparable = merged.dropna(subset=["pref_gpt4", "pref_claude"]).copy()
comparable["pref_claude"] = comparable["pref_claude"].astype(int)
comparable["pref_gpt4"]   = comparable["pref_gpt4"].astype(int)

print("Строк в каждом файле:", len(gpt4), len(claude))
print("Сопоставимых объектов:", len(comparable))
print("Доля исключённых:", round(1 - len(comparable)/len(gpt4), 4))
print("Значения preference Claude:", sorted(comparable['pref_claude'].unique()))
print("Значения preference GPT-4 :", sorted(comparable['pref_gpt4'].unique()))

Строки выровнены по содержимому пары: True
Строк в каждом файле: 10465 10465
Сопоставимых объектов: 10463
Доля исключённых: 0.0002
Значения preference Claude: [np.int64(0), np.int64(1), np.int64(2)]
Значения preference GPT-4 : [np.int64(0), np.int64(1), np.int64(2)]


In [2]:
# Находим строки, в которых отсутствует preference 
missing = (claude["preference"].isna() | gpt4["preference"].isna())

print("Строки, где отсутствует preference:", missing.sum())

print("- отсутствует Claude preference:", claude["preference"].isna().sum())
print("- отсутствует GPT-4 preference:", gpt4["preference"].isna().sum())
print("- отсутствуют оба:",(claude["preference"].isna() & gpt4["preference"].isna()).sum())

missing_objects = claude.loc[missing, KEY].copy()

print('Объекты, в которых отсутствует preference:')
missing_objects["pref_claude"] = (claude.loc[missing, "preference"].values)
missing_objects["pref_gpt4"] = (gpt4.loc[missing, "preference"].values)

display(missing_objects)

Строки, где отсутствует preference: 2
- отсутствует Claude preference: 0
- отсутствует GPT-4 preference: 2
- отсутствуют оба: 0
Объекты, в которых отсутствует preference:


,instruction,output_1,output_2,generator_1,generator_2,pref_claude,pref_gpt4
793,Solve for x in the equation 3x + 10 = 5(x - 2).,x = 8,x = 9,text_davinci_003,text_davinci_001,1,NaN
6589,Marie is at the music store for a store day. S...,Marie spent a total of $147.,Marie spent a total of $120.,text_davinci_003,wizardlm-13b,1,NaN


In [3]:
# Количество объектов в сопоставимом наборе
print('Количество объектов в сопоставимом наборе:', len(comparable))

Количество объектов в сопоставимом наборе: 10463


## Дополнительный шаг: категоризация исходных инструкций

В соответствии с  заявленным в карточке разобьем инструкций на категории. Задаем свои категории по содержанию инструкции до разметки:
- factual_qa — факты, определения, объяснения;
- creative_writing — тексты, истории, стихи, маркетинг;
- coding — код, программы, отладка;
- math_reasoning — математика, логика, пошаговый вывод;
- advice_howto — советы, инструкции «как сделать»;
- brainstorming_list — списки идей, перечисления;
- roleplay_dialogue — диалоги, ролевые сценарии;
- other.

In [4]:
import re

def categorize(instruction: str) -> str:
    t = instruction.lower()
    if any(k in t for k in ["code", "function", "python", "program", "bug", "sql", "javascript", "compile"]):
        return "coding"
    if any(k in t for k in ["math", "calculate", "equation", "solve", "sum", "probability", "integral"]):
        return "math_reasoning"
    if any(k in t for k in ["write a story", "poem", "essay", "slogan", "script", "dialogue", "roleplay", "act as"]):
        return "creative_writing"
    if any(k in t for k in ["how do i", "how to", "steps", "recipe", "instructions for", "guide"]):
        return "advice_howto"
    if any(k in t for k in ["list", "ideas", "brainstorm", "name some", "examples of", "suggest"]):
        return "brainstorming_list"
    if t.startswith("what") or t.startswith("why") or t.startswith("explain") or t.startswith("who") or t.startswith("when"):
        return "factual_qa"
    return "other"

comparable["category"] = comparable["instruction"].apply(categorize)
print(comparable["category"].value_counts())

category
other                 4835
factual_qa            1768
advice_howto           936
coding                 897
brainstorming_list     832
math_reasoning         610
creative_writing       585
Name: count, dtype: int64


In [5]:
# Удалим лишние столбцы 
comparable = comparable.drop(columns = ['annotator', 'preference', 'price_per_example',
       'time_per_example'])

In [6]:
# Сохраняем для дальнейшей работы
comparable.to_json('full_dataset.json')

## Итоговый набор для `slice.jsonl`

`slice.jsonl` — набор объектов для аудита. Он содержит 52 объекта из сопоставимой выборки AlpacaFarm: 20 объектов с `group="core"` и 32 объекта с `group="second"`. У каждого объекта есть уникальный `id`, поле `group`, категория исходной инструкции и содержимое, необходимое для независимой человеческой разметки: инструкция, два варианта ответа и названия генераторов. Каждый объект записан отдельной строкой JSON.

Объектом аудита является пара ответов `output_1` и `output_2` на одну инструкцию. В `slice.jsonl` не включаются оценки GPT-4 и Claude. Оценки моделей будем хранить отдельно в файлах `scorer_gpt4.jsonl` и `scorer_claude.jsonl`.

Для формирования выборки сначала сопоставляем записи из файлов оценок GPT-4 и Claude по инструкции и обоим вариантам ответа: `divergence = True`, если GPT-4 и Claude присвоили разные метки предпочтения, и `False`, если их метки совпали.

Далее выполняем квотную стратификацию по двум признакам:
- категории исходной инструкции;
- наличию расхождения между оценками GPT-4 и Claude.

Это позволяет сохранить содержательное разнообразие инструкций и включить в выборку как случаи согласия моделей, так и случаи их расхождения. Наша цель -- максимально репрезентативная выборка из текущего набора.

Квоты для итогового набора:

| Категория | Core | Second | Всего |
|---|---:|---:|---:|
| factual_qa | 4 | 6 | 10 |
| advice_howto | 3 | 5 | 8 |
| coding | 3 | 5 | 8 |
| creative_writing | 2 | 3 | 5 |
| math_reasoning | 2 | 3 | 5 |
| brainstorming_list | 2 | 3 | 5 |
| other | 4 | 7 | 11 |
| **Итого** | **20** | **32** | **52** |

In [7]:
import numpy as np
import json
import os

# Гарантируем, что индекс сопоставимой таблицы корректный
comparable = comparable.reset_index(drop=True)

# Двумерная переменная: расхождение между GPT-4 и Claude
comparable["divergence"] = comparable["pref_gpt4"] != comparable["pref_claude"]

print("Распределение по категориям и расхождениям:")
print(
    comparable
    .groupby(["category", "divergence"])
    .size()
    .unstack(fill_value=0)
)

# Квоты по категориям: core + second = 52 объекта
QUOTAS = {
    "factual_qa": {"core": 4, "second": 6},
    "advice_howto": {"core": 3, "second": 5},
    "coding": {"core": 3, "second": 5},
    "creative_writing": {"core": 2, "second": 3},
    "math_reasoning": {"core": 2, "second": 3},
    "brainstorming_list": {"core": 2, "second": 3},
    "other": {"core": 4, "second": 7},
}


def sample_with_divergence_balance(df, n_total, seed=2026):
    """
    Берет n_total объектов из одной категории.
    Сохраняет естественную пропорцию divergence,
    но гарантирует наличие обоих типов, если они доступны.
    """
    if len(df) == 0:
        return df.iloc[0:0].copy()

    n_total = min(n_total, len(df))

    div_true = df[df["divergence"]].copy()
    div_false = df[~df["divergence"]].copy()

    n_true_target = int(round(n_total * len(div_true) / len(df)))
    n_false_target = n_total - n_true_target

    n_true = min(n_true_target, len(div_true))
    n_false = min(n_false_target, len(div_false))

    # Если одного типа не хватает, добираем другим типом
    missing = n_total - n_true - n_false

    if missing > 0:
        available_true = len(div_true) - n_true
        available_false = len(div_false) - n_false

        if available_true >= missing:
            n_true += missing
        elif available_false >= missing:
            n_false += missing
        else:
            # Берем максимум из обоих типов
            add_true = min(available_true, missing)
            n_true += add_true
            missing -= add_true

            add_false = min(available_false, missing)
            n_false += add_false
            missing -= add_false

    selected_parts = []

    if n_true > 0:
        selected_parts.append(
            div_true.sample(n=n_true, random_state=seed)
        )

    if n_false > 0:
        selected_parts.append(
            div_false.sample(n=n_false, random_state=seed)
        )

    if not selected_parts:
        return df.iloc[0:0].copy()

    selected = pd.concat(selected_parts)

    # Если после округлений или нехватки объектов выбрали меньше,
    # добираем из оставшихся объектов категории
    remaining = n_total - len(selected)

    if remaining > 0:
        rest = df.drop(index=selected.index)

        if len(rest) > 0:
            extra = rest.sample(
                n=min(remaining, len(rest)),
                random_state=seed
            )
            selected = pd.concat([selected, extra])

    return selected.reset_index(drop=True)


core_parts = []
second_parts = []

for category, quotas in QUOTAS.items():
    category_df = comparable[
        comparable["category"] == category
    ].copy().reset_index(drop=True)

    n_core = quotas["core"]
    n_second = quotas["second"]

    # Если объектов в категории меньше, чем нужно,
    # уменьшаем квоты пропорционально
    available = len(category_df)

    if available < n_core + n_second:
        scale = available / (n_core + n_second)
        n_core = int(np.floor(n_core * scale))
        n_second = int(np.floor(n_second * scale))

        # Добираем остаток, чтобы не терять объекты
        while n_core + n_second < available:
            if n_core / max(n_core + n_second, 1) < 20 / 52:
                n_core += 1
            else:
                n_second += 1

    core_part = sample_with_divergence_balance(
        category_df,
        n_total=n_core,
        seed=2026
    )

    # Оставшиеся объекты категории
    remaining_category = category_df.drop(
        index=core_part.index,
        errors="ignore"
    ).reset_index(drop=True)

    second_part = sample_with_divergence_balance(
        remaining_category,
        n_total=n_second,
        seed=2026
    )

    core_parts.append(core_part)
    second_parts.append(second_part)

core_sample = pd.concat(core_parts, ignore_index=True)
second_sample = pd.concat(second_parts, ignore_index=True)

# Если из-за нехватки объектов в каких-то категориях набралось меньше,
# добираем случайные объекты из общего пула
def top_up_sample(current, target_n, pool, seed=2026):
    missing = target_n - len(current)

    if missing <= 0:
        return current

    used_ids = set(current.index) if len(current) else set()
    available = pool.drop(index=used_ids, errors="ignore")

    if len(available) == 0:
        return current

    extra = available.sample(
        n=min(missing, len(available)),
        random_state=seed
    )

    return pd.concat([current, extra], ignore_index=True)


# Объекты, еще не попавшие в текущие выборки
all_selected = pd.concat([core_sample, second_sample], ignore_index=True)

# Добор только при необходимости
core_sample = top_up_sample(core_sample, target_n=20, 
                            pool=comparable.drop(index=all_selected.index, errors="ignore"), seed=2026)

second_sample = top_up_sample(second_sample, target_n=32,
                              pool=comparable.drop(index=all_selected.index, errors="ignore"), seed=2026)

# Проверки
assert len(core_sample) == 20, f"Core: {len(core_sample)}"
assert len(second_sample) == 32, f"Second: {len(second_sample)}"
assert not core_sample.index.duplicated().any()
assert not second_sample.index.duplicated().any()

print("\nCore:")
print(core_sample["category"].value_counts())
print(core_sample["divergence"].value_counts())

print("\nSecond:")
print(second_sample["category"].value_counts())
print(second_sample["divergence"].value_counts())


# Собираем итоговые 52 объекта
def to_obj(r, group, i):
    obj = {
        "id": f"ALP-{i:04d}",
        "group": group,
        "category": str(r["category"]),
        "instruction": r["instruction"],
        "output_1": r["output_1"],
        "generator_1": r["generator_1"],
        "output_2": r["output_2"],
        "generator_2": r["generator_2"]
    }


    obj["dataset"] = str(r["dataset"])

    return obj


rows = []

for i, (_, r) in enumerate(core_sample.iterrows(), start=1):
    rows.append(to_obj(r, "core", i))

for i, (_, r) in enumerate(second_sample.iterrows(), start=21):
    rows.append(to_obj(r, "second", i))

# Сохраняем slice.jsonl
os.makedirs("audit", exist_ok=True)

with open("audit/slice.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\nЗаписано объектов: {len(rows)}")
print(f"Core: {sum(r['group'] == 'core' for r in rows)}")
print(f"Second: {sum(r['group'] == 'second' for r in rows)}")

Распределение по категориям и расхождениям:


divergence          False  True 
category                        
advice_howto          702    234
brainstorming_list    639    193
coding                679    218
creative_writing      452    133
factual_qa           1377    391
math_reasoning        461    149
other                3682   1153

Core:
category
factual_qa            4
other                 4
advice_howto          3
coding                3
creative_writing      2
math_reasoning        2
brainstorming_list    2
Name: count, dtype: int64
divergence
False    16
True      4
Name: count, dtype: int64

Second:
category
other                 7
factual_qa            6
advice_howto          5
coding                5
creative_writing      3
math_reasoning        3
brainstorming_list    3
Name: count, dtype: int64
divergence
False    24
True      8
Name: count, dtype: int64

Записано объектов: 52
Core: 20
Second: 32
